# Feature Engineering — RFM & Behavioral Features

This notebook builds the feature matrix used for **customer churn prediction**.  
We compute RFM (Recency, Frequency, Monetary) features, behavioural signals
(review scores, delivery deltas, payment habits), create the churn label, and
perform feature-level diagnostics before model training.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.figure_factory as ff

# Dark-theme colour palette
PAPER_BG  = '#0D1117'
PLOT_BG   = '#0D1117'
FONT_CLR  = '#94A3B8'
ACCENT    = '#00D4B1'

PLOTLY_LAYOUT = dict(
    paper_bgcolor=PAPER_BG,
    plot_bgcolor=PLOT_BG,
    font=dict(color=FONT_CLR),
)

print('Imports OK ✓')

In [ ]:
from src.data_loader import load_all_datasets, merge_master_dataframe

datasets = load_all_datasets(data_dir=PROJECT_ROOT / 'data' / 'raw')
df = merge_master_dataframe(datasets)

# Ensure datetime columns are parsed
date_cols = [
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

print(f'Master dataframe shape: {df.shape}')
df.head()

## RFM Features

Compute **Recency** (days since last purchase), **Frequency** (number of orders),
and **Monetary** (total spend) per customer.

In [ ]:
from src.feature_engineering import compute_rfm_features

rfm = compute_rfm_features(df)
print(f'RFM shape: {rfm.shape}')
rfm.head(10)

## Behavioral Features

Extract behavioural signals such as average review score, delivery deviation,
preferred payment method, number of product categories ordered, etc.

In [ ]:
from src.feature_engineering import compute_behavioral_features

behavioral = compute_behavioral_features(df)
print(f'Behavioral features shape: {behavioral.shape}')
behavioral.head(10)

## Churn Label

Define **churn** as a customer who has *not* placed an order in the last 90 days
relative to the dataset's latest order date.  
- `churn = 1` → churned  
- `churn = 0` → active / retained

In [ ]:
CHURN_THRESHOLD_DAYS = 90

max_date = df['order_purchase_timestamp'].max()

last_purchase = (
    df.groupby('customer_unique_id')['order_purchase_timestamp']
    .max()
    .reset_index()
    .rename(columns={'order_purchase_timestamp': 'last_purchase_date'})
)
last_purchase['days_since_last'] = (max_date - last_purchase['last_purchase_date']).dt.days
last_purchase['churn'] = (last_purchase['days_since_last'] >= CHURN_THRESHOLD_DAYS).astype(int)

print('Churn label distribution:')
print(last_purchase['churn'].value_counts())
print(f'\nChurn rate: {last_purchase["churn"].mean():.2%}')

## Full Feature Matrix

Merge RFM + behavioural features + churn label into a single feature matrix.

In [ ]:
feature_matrix = (
    rfm
    .merge(behavioral, on='customer_unique_id', how='left')
    .merge(last_purchase[['customer_unique_id', 'churn']], on='customer_unique_id', how='left')
)

print(f'Feature matrix shape: {feature_matrix.shape}')
print()
print('=== Descriptive Statistics ===')
display(feature_matrix.describe())
print()
print('=== Missing Values ===')
nan_counts = feature_matrix.isnull().sum()
print(nan_counts[nan_counts > 0] if nan_counts.any() else 'No missing values ✓')

In [ ]:
# Feature distributions — histograms for numeric columns
numeric_cols = feature_matrix.select_dtypes(include='number').columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'churn']  # exclude label

for col in numeric_cols:
    fig = px.histogram(
        feature_matrix, x=col, nbins=50,
        title=f'Distribution of {col}',
        color_discrete_sequence=[ACCENT],
    )
    fig.update_layout(**PLOTLY_LAYOUT)
    fig.show()

In [ ]:
# Correlation heatmap
corr = feature_matrix[numeric_cols + ['churn']].corr()

fig = px.imshow(
    corr,
    text_auto='.2f',
    color_continuous_scale='Tealgrn',
    title='Feature Correlation Heatmap',
    aspect='auto',
)
fig.update_layout(**PLOTLY_LAYOUT)
fig.show()

## Summary & Next Steps

| Step | Status |
|------|--------|
| RFM features computed | ✅ |
| Behavioral features computed | ✅ |
| Churn label created (90-day threshold) | ✅ |
| Feature matrix assembled & validated | ✅ |
| Feature distributions inspected | ✅ |
| Correlation analysis completed | ✅ |

**Key observations:**
- Recency is the strongest correlate of churn (expected by construction).
- Some features may need log-transformation due to right-skew.
- No critical missing-value issues after the merge.

---
*Next → `03_churn_model.ipynb` — Train & evaluate churn prediction models.*